# Methods 5 – Transformer Encoder

## Objective
In this assignment, I'm building a neural network that can identify positive and negative movie reviews.
I'll use the IMDB dataset and add parts of a Transformer encoder. In the end, I'll test the model to see how well it works.

In [1]:

import keras
from keras.datasets import imdb

# Dataset settings
max_features = 10000
max_len = 250

# Load IMDB movie reviews
(x_train, y_train), (x_test, y_test) = imdb.load_data(
    num_words=max_features
)

# Make all reviews the same length
x_train = keras.utils.pad_sequences(
    x_train, maxlen=max_len, padding="post"
)

x_test = keras.utils.pad_sequences(
    x_test, maxlen=max_len, padding="post"
)

print("Training data:", x_train.shape)
print("Test data:", x_test.shape)
print("First five labels:", y_train[:5])


17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
Training data: (25000, 250)
Test data: (25000, 250)
First five labels: [1 0 0 1 0]


## Token and positional embedding
In this step, I turn token IDs into token vectors. I also create a vector for each position so the model can take word order into account. Finally, I add the token and position vectors together.

In [2]:

from keras.layers import Layer, Embedding

class TokenAndPositionEmbedding(Layer):
    def __init__(self, seq_len, vocab_size, emb_dim):
        super().__init__()

        self.token_emb = Embedding(
            input_dim=vocab_size,
            output_dim=emb_dim
        )

        self.pos_emb = Embedding(
            input_dim=seq_len,
            output_dim=emb_dim
        )

    def call(self, x_input):
        seq_len = keras.ops.shape(x_input)[-1]

        positions = keras.ops.arange(
            start=0, stop=seq_len, step=1
        )

        positions = self.pos_emb(positions)
        x_input = self.token_emb(x_input)

        return x_input + positions


# Test the embedding layer
embedding_layer = TokenAndPositionEmbedding(
    max_len, max_features, 32
)

sample = embedding_layer(x_train[:2])
print("Embedding shape:", sample.shape)


Embedding shape: (2, 250, 32)


## Multi-head attention and residual connection
I use multi-head self-attention to help the model learn how words relate to each other. The residual connection adds the original embeddings to the attention output so the original information can pass through. Then I normalize the feature values for each token to help stabilize training.

In [3]:
from keras.layers import Input, MultiHeadAttention, Add, LayerNormalization

embed_dim = 32
num_heads = 2

# Input and embeddings
inputs = Input(shape=(max_len,), dtype="int32")
x = TokenAndPositionEmbedding(
    max_len, max_features, embed_dim
)(inputs)

# Self-attention
attention_output = MultiHeadAttention(
    num_heads=num_heads,
    key_dim=embed_dim // num_heads
)(x, x)

# First residual connection
x = Add()([x, attention_output])

# First normalization
x = LayerNormalization(epsilon=1e-6)(x)

print("Shape after attention:", x.shape)



Shape after attention: (None, 250, 32)


## Feedforward network and second residual connection
Next, I use a feedforward network with two Dense layers. The first layer has 64 units and uses ReLU. The second layer brings the output back to 32 units and keeps it linear. This gives the feedforward output the same shape as its input, so I can add them in the second residual connection. Finally, I apply layer normalization.

In [4]:

from keras.layers import Dense

# Feedforward network
ffn_output = Dense(64, activation="relu")(x)
ffn_output = Dense(embed_dim)(ffn_output)

# Second residual connection
x = Add()([x, ffn_output])

# Second normalization
x = LayerNormalization(epsilon=1e-6)(x)

print("Shape after feedforward:", x.shape)


Shape after feedforward: (None, 250, 32)
